# Customer Churn Prediction.

## Data Preprocessing.

En esta fase del proyecto se realiza el preprocesamiento de los datos con el objetivo de preparar el dataset para el entrenamiento de modelos de machine learning.

Las principales tareas incluyen:

- Transformación de variables categóricas.
- Análisis de multicolinealidad.
- Escalado de variables numéricas.
- Separación entre variables predictoras y variable objetivo.
- División en conjuntos de entrenamiento y prueba.

## Importamos librerías.

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

# Carga del dataset.

In [2]:
df = pd.read_csv(r"..\Data\Processed\telco_churn_eda_completed.csv")

# Vista general del dataset.

In [3]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [5]:
df.shape

(7043, 20)

# Revisión de valores nulos.

Verifica nuevamente la existencia de valores nulos en el dataset antes de comenzar el preprocesamiento.

In [6]:
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

# Transformación de variables categóricas.

Los algoritmos de machine learning requieren variables numéricas, por lo que las variables categóricas deben transformarse antes del entrenamiento.

## Identificación de variables categóricas.

In [7]:
categorical_columns = df.select_dtypes(include='object').columns

categorical_columns

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

## Aplicación de One-Hot Encoding.
Aplicamos One-Hot Encoding para transformar las variables categóricas en variables binarias.

In [8]:
df = pd.get_dummies(df, drop_first=True)

# Verificación del dataset transformado.

In [9]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False


In [10]:
df.shape

(7043, 31)

# Análisis de multicolinealidad.

Durante el análisis exploratorio se detectó una correlación elevada entre `tenure` y `TotalCharges`.

La multicolinealidad puede afectar especialmente a modelos lineales como la regresión logística, generando coeficientes inestables y dificultando la interpretación de la importancia individual de las variables.

No obstante, una alta correlación entre variables no implica necesariamente que deban eliminarse automáticamente, especialmente si ambas aportan información relevante desde una perspectiva de negocio.

## Correlación entre tenure y TotalCharges.

In [11]:
df[['tenure', 'TotalCharges']].corr()

,tenure,TotalCharges
tenure,1.000000,0.826178
TotalCharges,0.826178,1.000000


Se observa una correlación positiva elevada entre `tenure` y `TotalCharges`.

Este comportamiento resulta coherente desde una perspectiva de negocio, ya que clientes con mayor antigüedad tienden a acumular un gasto total más elevado a lo largo del tiempo.

Aunque esta relación podría generar problemas de multicolinealidad en algunos modelos lineales, ambas variables se mantienen inicialmente debido a su capacidad explicativa.

El posible impacto de esta correlación se evaluará posteriormente durante la fase de modelado.

# Separación entre variables predictoras y variable objetivo.

In [12]:
X = df.drop('Churn', axis=1)

y = df['Churn']

# División en train y test.

El dataset se divide en conjuntos de entrenamiento y prueba para evaluar posteriormente la capacidad de generalización de los modelos.

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=72,
    stratify=y
)

## Verificación de dimensiones.

In [14]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5634, 30)
(1409, 30)
(5634,)
(1409,)


# Escalado de variables numéricas.

Algunos algoritmos de machine learning son sensibles a la escala de las variables, especialmente modelos lineales y métodos basados en distancia.

Por este motivo, se aplica estandarización mediante `StandardScaler`.

## Identificación de variables numéricas.

In [15]:
numerics_columns = X_train.select_dtypes(include=['int64', 'float64']).columns

numerics_columns

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

## Aplicación de StandardScaler.

In [16]:
scaler =StandardScaler()

X_train[numerics_columns] = scaler.fit_transform(
    X_train[numerics_columns]
)

X_test[numerics_columns] = scaler.transform(
    X_test[numerics_columns]
)

# Verificación del escalado.

In [17]:
X_train.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
97,-0.439763,-1.121321,-1.465720,-0.962443,True,False,False,True,False,False,False,True,True,False,True,False,True,False,True,False,True,False,True,False,False,False,False,False,False,True
669,-0.439763,1.523560,-0.241169,0.759627,False,True,False,False,True,False,False,False,False,False,False,True,False,True,False,True,False,True,False,True,False,True,False,False,False,False
1190,-0.439763,0.221465,1.326589,0.692923,True,False,False,True,False,True,True,False,False,True,False,True,False,False,False,False,False,True,False,True,True,False,True,False,True,False
1472,-0.439763,-0.795797,0.311961,-0.618990,True,True,True,True,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False
2996,-0.439763,-0.592345,-0.147870,-0.505105,False,False,False,True,False,True,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True


# Guardado de datasets procesados.

Guardamos los conjuntos de entrenamiento y prueba tras el preprocesamiento para asegurar la trazabilidad y reutilización en la fase de modelado.

In [18]:
os.makedirs(
    r"..\Data\Processed",
    exist_ok=True
)

X_train.to_csv(
    r"..\Data\Processed\X_train.csv",
    index=False
)

X_test.to_csv(
    r"..\Data\Processed\X_test.csv",
    index=False
)

y_train.to_csv(
    r"..\Data\Processed\y_train.csv",
    index=False
)

y_test.to_csv(
    r"..\Data\Processed\y_test.csv",
    index=False
)

# Conclusiones.

En esta fase del proyecto se prepararon los datos para el entrenamiento de modelos de machine learning.

Se transformaron las variables categóricas mediante One-Hot Encoding, se analizó la posible multicolinealidad entre variables numéricas y se aplicó escalado mediante `StandardScaler`.

Finalmente, el dataset fue dividido en conjuntos de entrenamiento y prueba, dejando los datos preparados para la fase de modelado predictivo.